In [1]:
pip install ydata-profiling[pyspark]

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession
from ydata_profiling import ProfileReport

In [3]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Python Spark profiling example") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

In [4]:
# Copy File to bronze layer
from os import PathLike
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

from_path = "./amazon_credits.csv"
to_path = "/demo/bronze/amazon_credits.csv"

client.delete(to_path)
client.upload(to_path, from_path)

'/demo/bronze/amazon_credits.csv'

In [5]:
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/amazon_credits.csv"

In [6]:
# Create a DataFrame from JSON data (automatically infer schema and data types)
# There are other file formats you can read from (e.g., csv, orc, parquet)
# https://spark.apache.org/docs/2.2.0/sql-programming-guide.html#data-sources

# Read Oscar data to a dataframe
amazon_credits_df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

In [13]:
amazon_credits_df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

In [14]:
# Note that some profiling operations can resulte in errors due to bad loading options. 
# It is a good praticce start by inspect the schema and a data sample. 
amazon_credits_df.printSchema()
amazon_credits_df.show()

root
 |-- person_id: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- character: string (nullable = true)
 |-- role: string (nullable = true)

+---------+-------+-------------------+--------------------+-----+
|person_id|     id|               name|           character| role|
+---------+-------+-------------------+--------------------+-----+
|    59401|ts20945|         Joe Besser|                 Joe|ACTOR|
|    31460|ts20945|         Moe Howard|                 Moe|ACTOR|
|    31461|ts20945|         Larry Fine|               Larry|ACTOR|
|    21174|tm19248|      Buster Keaton|         Johnny Gray|ACTOR|
|    28713|tm19248|        Marion Mack|       Annabelle Lee|ACTOR|
|    28714|tm19248|      Glen Cavender|    Captain Anderson|ACTOR|
|    28715|tm19248|         Jim Farley|    General Thatcher|ACTOR|
|    27348|tm19248|    Frederick Vroom|  A Southern General|ACTOR|
|    28716|tm19248|Charles Henry Smith|  Annabelle's Father|ACTOR|
|

In [15]:
amazon_credits_df.select("role").distinct().count()

2

In [31]:
amazon_credits_df.select("character").distinct().count()

71098

In [32]:
amazon_credits_df.filter(F.col("role") == "DIRECTOR").count()

8389

In [26]:
from pyspark.sql import functions as F

# número total de linhas
total_rows = amazon_credits_df.count()

# para cada coluna, contar valores repetidos
for col in amazon_credits_df.columns:
    distinct_count = amazon_credits_df.select(col).distinct().count()
    repeated_count = total_rows - distinct_count
    print(f"{col}: {repeated_count} valores repetidos")

person_id: 43727 valores repetidos
id: 115374 valores repetidos
name: 44477 valores repetidos
character: 53137 valores repetidos
role: 124233 valores repetidos


In [27]:
from pyspark.sql import functions as F

for col in amazon_credits_df.columns:
    blank_count = amazon_credits_df.filter(
        (F.col(col).isNull()) | (F.col(col) == "") | (F.col(col).isin(" ", "null", "None"))
    ).count()
    print(f"{col}: {blank_count} valores vazios")

person_id: 0 valores vazios
id: 0 valores vazios
name: 0 valores vazios
character: 16287 valores vazios
role: 0 valores vazios


In [24]:
from pyspark.sql import functions as F
min_id = amazon_credits_df.agg(F.min("person_id").alias("min_id")).collect()[0]["min_id"]
amazon_credits_df.select(F.min("person_id"), F.max("person_id")).show()

+--------------+--------------+
|min(person_id)|max(person_id)|
+--------------+--------------+
|             1|       2371153|
+--------------+--------------+



In [17]:
# In case of error select a subset of columns until you find the column that causes that.
#For start we can use describe as starting point for data profiling
#For this example the column summary was removed due to a conflit with the first describe column "summary"
amazon_credits_df.describe(['person_id','id','name','character','role']).toPandas()

,summary,person_id,id,name,character,role
0,count,124235,124235,124235,107948,124235
1,mean,406473.6839859943,None,None,NaN,None
2,stddev,561629.646076384,None,None,NaN,None
3,min,1,tm100001,Nelson Venkatesan,"""44""",ACTOR
4,max,2371153,ts9913,윤도,새미,DIRECTOR


In [33]:
#Select the columns to profile. 
df_to_profile = amazon_credits_df.select("person_id","id","name","character","role")

In [34]:
#create the Profile report using the ydata profiling tool. More info at https://docs.profiling.ydata.ai/
report = ProfileReport(df_to_profile)

In [35]:
#save profiling report in a file
report.to_file('amazon_credits.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
#close spark session
spark.stop()